# Fine-tune Cross-Encoder v0.6 - Model Selection (Separate Cells)

Phase 3: Test 3 different base models with independent cells. Each cell trains ONE model - if one fails, run others separately.

| | |
|---|---|
| **Dataset** | v0.5 — 9,350 train / 2,000 validation / 2,000 test (13,350 pairs, balanced 20% per class) |
| **Loss** | `MSELoss` |
| **Evaluator** | Spearman correlation |
| **Epochs** | **15** (fixed) |
| **Learning rate** | **5e-5** (locked from Phase 2.1) |
| **Batch size** | **16** (locked from baseline) |
| **Branch** | `experiment/cross-encoder-v0.6` |

**Goal**: Compare 3 architectures. Run cells independently (if cell 2 fails, you can still run cells 1 & 3).

**Expected**: +1-3pp improvement over baseline 65.15% → 66-68% target.

In [ ]:
import os
if not os.path.exists('/content/Ai-Recruiter-Mini-Ai-Service'):
    !git clone https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service /content/Ai-Recruiter-Mini-Ai-Service

%cd /content/Ai-Recruiter-Mini-Ai-Service
!git checkout experiment/cross-encoder-v0.6
!git pull origin experiment/cross-encoder-v0.6

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Helper Functions & Data Load

In [ ]:
import json
import torch
import numpy as np
from scipy.stats import spearmanr
from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader
from typing import Any

# Evaluator: CECorrelationEvaluator (Spearman)
class CECorrelationEvaluator:
    """Evaluate cross-encoder using Spearman correlation."""
    def __init__(self, sentence_pairs: list, labels_0_1: list[float], name: str = ""):
        self.sentence_pairs = sentence_pairs
        self.labels_0_1 = labels_0_1
        self.name = name

    @classmethod
    def from_input_examples(cls, examples, name: str = "") -> "CECorrelationEvaluator":
        return cls(
            sentence_pairs=[ex.texts for ex in examples],
            labels_0_1=[ex.label for ex in examples],
            name=name,
        )

    def __call__(self, model, output_path=None, epoch: int = -1, steps: int = -1) -> float:
        preds = model.predict(self.sentence_pairs, batch_size=32, show_progress_bar=False)
        corr, _ = spearmanr(preds, self.labels_0_1)
        return float(corr) if not np.isnan(corr) else 0.0

def load_jsonl(path: str) -> list[dict[str, Any]]:
    """Load JSONL file."""
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def load_dataset(data_dir: str) -> tuple[list[InputExample], list[InputExample], list[InputExample]]:
    """Load train/val/test InputExample lists."""
    train_data = load_jsonl(f"{data_dir}/cross_encoder_train.jsonl")
    val_data = load_jsonl(f"{data_dir}/cross_encoder_validation.jsonl")
    test_data = load_jsonl(f"{data_dir}/cross_encoder_test.jsonl")

    def to_input_examples(records):
        return [
            InputExample(texts=[rec['cv_text'], rec['jd_text']], label=rec['label'])
            for rec in records
        ]

    return to_input_examples(train_data), to_input_examples(val_data), to_input_examples(test_data)

def compute_metrics(model: CrossEncoder, examples: list[InputExample], batch_size: int = 32) -> dict:
    """Compute MAE, RMSE, LabelAcc on examples."""
    preds = model.predict([ex.texts for ex in examples], batch_size=batch_size, show_progress_bar=False)
    preds_100 = np.asarray(preds) * 100
    labels_100 = np.array([ex.label * 100 for ex in examples])

    mae = np.mean(np.abs(preds_100 - labels_100))
    rmse = np.sqrt(np.mean((preds_100 - labels_100) ** 2))
    label_acc = np.mean(np.abs(preds_100 - labels_100) <= 10)

    return {'MAE': mae, 'RMSE': rmse, 'LabelAcc': label_acc}

# Load dataset once
train_examples, val_examples, test_examples = load_dataset('datasets/versions/v0.5/cross_encoder')

print(f"✅ Helpers loaded")
print(f"✅ Dataset loaded: {len(train_examples)} train, {len(val_examples)} val, {len(test_examples)} test")

# Fixed hyperparameters
LR = 5e-5
BS = 16
EPOCHS = 15
TOTAL_STEPS = (len(train_examples) // BS + 1) * EPOCHS
WARMUP_STEPS = int(TOTAL_STEPS * 0.1)

print(f"\n📊 Fixed Hyperparameters:")
print(f"  LR: {LR:.0e}")
print(f"  BS: {BS}")
print(f"  Epochs: {EPOCHS}")
print(f"  Warmup: {WARMUP_STEPS} (10% of {TOTAL_STEPS})")

## Model 1: ms-marco-MiniLM-L-12-v2 (Baseline)

**Independent Cell** - Run this separately. If it fails, don't affect other models.

In [ ]:
print(f"\n{'='*70}")
print(f"🚀 Model 1: ms-marco-MiniLM-L-12-v2 (Baseline)")
print(f"   12-layer, 384-dim, English-only")
print(f"{'='*70}")

model1_name = 'ms-marco-MiniLM-L-12-v2'
model1_hf = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
model1_run = f"v0.6-{model1_name}-mse-spearman-lr5e-05-bs16-15ep"
model1_dir = f"artifacts/models/cross-encoder-cv-jd-{model1_run}"

# Initialize
model1 = CrossEncoder(
    model1_hf,
    num_labels=1,
    default_activation_function=torch.nn.Sigmoid()
)

# Train
evaluator = CECorrelationEvaluator.from_input_examples(val_examples, name="val_spearman")
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BS)

model1.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=EPOCHS,
    loss_fct=torch.nn.MSELoss(),
    optimizer_params={'lr': LR},
    warmup_steps=WARMUP_STEPS,
    output_path=model1_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True
)

# Evaluate
best_model1 = CrossEncoder(model1_dir)
val1_metrics = compute_metrics(best_model1, val_examples)
test1_metrics = compute_metrics(best_model1, test_examples)

print(f"\n✅ Model 1 Completed")
print(f"  Val LabelAcc:  {val1_metrics['LabelAcc']:.4f} ({val1_metrics['LabelAcc']*100:.2f}%)")
print(f"  Test LabelAcc: {test1_metrics['LabelAcc']:.4f} ({test1_metrics['LabelAcc']*100:.2f}%)")
print(f"  vs Baseline (65.15%): {(test1_metrics['LabelAcc']-0.6515)*100:+.2f}pp")

# Store
result1 = {
    'model': model1_name,
    'hf_name': model1_hf,
    'run': model1_run,
    'val': val1_metrics,
    'test': test1_metrics
}

# Save individual report
import os
os.makedirs('artifacts/reports', exist_ok=True)

report1 = {
    'base_model': model1_hf,
    'dataset_version': 'v0.5',
    'dataset_size': {'train': len(train_examples), 'val': len(val_examples), 'test': len(test_examples)},
    'run': result1['run'],
    'loss': 'MSE',
    'evaluator': 'Spearman',
    'learning_rate': float(LR),
    'batch_size': int(BS),
    'epochs': EPOCHS,
    'metrics': {
        'validation': {k: float(v) for k, v in val1_metrics.items()},
        'test': {k: float(v) for k, v in test1_metrics.items()}
    },
    'model_path': model1_dir
}

with open('artifacts/reports/fine_tune_cross_encoder_v0.6_model_ms_marco_minilm_l12_report.json', 'w') as f:
    json.dump(report1, f, indent=2)

print(f"✅ Report saved: fine_tune_cross_encoder_v0.6_model_ms_marco_minilm_l12_report.json")

## Model 2: ms-marco-MiniLM-L-6-v2 (Lightweight)

**Independent Cell** - Run this separately. If Model 1 succeeded, this won't affect it.

In [ ]:
print(f"\n{'='*70}")
print(f"🚀 Model 2: ms-marco-MiniLM-L-6-v2 (Lightweight)")
print(f"   6-layer, 384-dim, faster")
print(f"{'='*70}")

model2_name = 'ms-marco-MiniLM-L-6-v2'
model2_hf = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
model2_run = f"v0.6-{model2_name}-mse-spearman-lr5e-05-bs16-15ep"
model2_dir = f"artifacts/models/cross-encoder-cv-jd-{model2_run}"

# Initialize
model2 = CrossEncoder(
    model2_hf,
    num_labels=1,
    default_activation_function=torch.nn.Sigmoid()
)

# Train
evaluator = CECorrelationEvaluator.from_input_examples(val_examples, name="val_spearman")
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BS)

model2.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=EPOCHS,
    loss_fct=torch.nn.MSELoss(),
    optimizer_params={'lr': LR},
    warmup_steps=WARMUP_STEPS,
    output_path=model2_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True
)

# Evaluate
best_model2 = CrossEncoder(model2_dir)
val2_metrics = compute_metrics(best_model2, val_examples)
test2_metrics = compute_metrics(best_model2, test_examples)

print(f"\n✅ Model 2 Completed")
print(f"  Val LabelAcc:  {val2_metrics['LabelAcc']:.4f} ({val2_metrics['LabelAcc']*100:.2f}%)")
print(f"  Test LabelAcc: {test2_metrics['LabelAcc']:.4f} ({test2_metrics['LabelAcc']*100:.2f}%)")
print(f"  vs Baseline (65.15%): {(test2_metrics['LabelAcc']-0.6515)*100:+.2f}pp")

# Store
result2 = {
    'model': model2_name,
    'hf_name': model2_hf,
    'run': model2_run,
    'val': val2_metrics,
    'test': test2_metrics
}

# Save individual report
report2 = {
    'base_model': model2_hf,
    'dataset_version': 'v0.5',
    'dataset_size': {'train': len(train_examples), 'val': len(val_examples), 'test': len(test_examples)},
    'run': result2['run'],
    'loss': 'MSE',
    'evaluator': 'Spearman',
    'learning_rate': float(LR),
    'batch_size': int(BS),
    'epochs': EPOCHS,
    'metrics': {
        'validation': {k: float(v) for k, v in val2_metrics.items()},
        'test': {k: float(v) for k, v in test2_metrics.items()}
    },
    'model_path': model2_dir
}

with open('artifacts/reports/fine_tune_cross_encoder_v0.6_model_ms_marco_minilm_l6_report.json', 'w') as f:
    json.dump(report2, f, indent=2)

print(f"✅ Report saved: fine_tune_cross_encoder_v0.6_model_ms_marco_minilm_l6_report.json")

## Model 3: qnli-distilroberta-base (Different Architecture)

**Independent Cell** - Run this separately. Different architecture from models 1-2.

In [ ]:
print(f"\n{'='*70}")
print(f"🚀 Model 3: qnli-distilroberta-base (Different Architecture)")
print(f"   6-layer DistilRoBERTa, 768-dim")
print(f"{'='*70}")

model3_name = 'qnli-distilroberta-base'
model3_hf = 'cross-encoder/qnli-distilroberta-base'
model3_run = f"v0.6-{model3_name}-mse-spearman-lr5e-05-bs16-15ep"
model3_dir = f"artifacts/models/cross-encoder-cv-jd-{model3_run}"

# Initialize
model3 = CrossEncoder(
    model3_hf,
    num_labels=1,
    default_activation_function=torch.nn.Sigmoid()
)

# Train
evaluator = CECorrelationEvaluator.from_input_examples(val_examples, name="val_spearman")
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BS)

model3.fit(
    train_dataloader=train_dataloader,
    evaluator=evaluator,
    epochs=EPOCHS,
    loss_fct=torch.nn.MSELoss(),
    optimizer_params={'lr': LR},
    warmup_steps=WARMUP_STEPS,
    output_path=model3_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True
)

# Evaluate
best_model3 = CrossEncoder(model3_dir)
val3_metrics = compute_metrics(best_model3, val_examples)
test3_metrics = compute_metrics(best_model3, test_examples)

print(f"\n✅ Model 3 Completed")
print(f"  Val LabelAcc:  {val3_metrics['LabelAcc']:.4f} ({val3_metrics['LabelAcc']*100:.2f}%)")
print(f"  Test LabelAcc: {test3_metrics['LabelAcc']:.4f} ({test3_metrics['LabelAcc']*100:.2f}%)")
print(f"  vs Baseline (65.15%): {(test3_metrics['LabelAcc']-0.6515)*100:+.2f}pp")

# Store
result3 = {
    'model': model3_name,
    'hf_name': model3_hf,
    'run': model3_run,
    'val': val3_metrics,
    'test': test3_metrics
}

# Save individual report
report3 = {
    'base_model': model3_hf,
    'dataset_version': 'v0.5',
    'dataset_size': {'train': len(train_examples), 'val': len(val_examples), 'test': len(test_examples)},
    'run': result3['run'],
    'loss': 'MSE',
    'evaluator': 'Spearman',
    'learning_rate': float(LR),
    'batch_size': int(BS),
    'epochs': EPOCHS,
    'metrics': {
        'validation': {k: float(v) for k, v in val3_metrics.items()},
        'test': {k: float(v) for k, v in test3_metrics.items()}
    },
    'model_path': model3_dir
}

with open('artifacts/reports/fine_tune_cross_encoder_v0.6_model_qnli_distilroberta_report.json', 'w') as f:
    json.dump(report3, f, indent=2)

print(f"✅ Report saved: fine_tune_cross_encoder_v0.6_model_qnli_distilroberta_report.json")

## Summary (Run after all 3 models completed)

In [ ]:
import pandas as pd

# Combine all results (only works if all 3 cells ran successfully)
all_results = [result1, result2, result3]

summary_data = []
for result in all_results:
    summary_data.append({
        'Model': result['model'],
        'Val LabelAcc': f"{result['val']['LabelAcc']:.4f}",
        'Test LabelAcc': f"{result['test']['LabelAcc']:.4f}",
        'vs Baseline': f"{(result['test']['LabelAcc']-0.6515)*100:+.2f}pp"
    })

df = pd.DataFrame(summary_data)
print("\n📊 Phase 3 - Model Selection Results:")
print(df.to_string(index=False))

# Find best
best_result = max(all_results, key=lambda x: x['test']['LabelAcc'])
print(f"\n🏆 Best Model: {best_result['model']}")
print(f"   Test LabelAcc: {best_result['test']['LabelAcc']:.4f} ({best_result['test']['LabelAcc']*100:.2f}%)")
print(f"   Improvement: {(best_result['test']['LabelAcc']-0.6515)*100:+.2f}pp over baseline 65.15%")

# Save summary
summary_report = {
    'experiment': 'Phase 3: Model Selection',
    'baseline': {'model': 'ms-marco-MiniLM-L-12-v2', 'test_label_acc': 0.6515},
    'results': all_results,
    'best': {
        'model': best_result['model'],
        'test_label_acc': float(best_result['test']['LabelAcc']),
        'improvement_pp': float((best_result['test']['LabelAcc']-0.6515)*100)
    }
}

with open('artifacts/reports/fine_tune_cross_encoder_v0.6_model_selection_summary.json', 'w') as f:
    json.dump(summary_report, f, indent=2)

print(f"\n✅ Summary saved: fine_tune_cross_encoder_v0.6_model_selection_summary.json")

## Save to Google Drive (Optional)

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

drive_base = "/content/drive/MyDrive/ai-recruiter"
os.makedirs(f"{drive_base}/models", exist_ok=True)
os.makedirs(f"{drive_base}/reports", exist_ok=True)

# Save all models (if they exist)
for result in [result1, result2, result3]:
    src_dir = f'artifacts/models/cross-encoder-cv-jd-{result["run"]}'
    if os.path.exists(src_dir):
        dest_dir = f"{drive_base}/models/cross-encoder-cv-jd-{result['run']}"
        if os.path.exists(dest_dir):
            shutil.rmtree(dest_dir)
        shutil.copytree(src_dir, dest_dir)
        print(f"✅ Saved model: {result['run']}")
    else:
        print(f"⚠️  Model not found: {src_dir}")

# Copy reports
for file in os.listdir('artifacts/reports'):
    if 'model_' in file and 'selection' in file:
        src = f'artifacts/reports/{file}'
        dest = f"{drive_base}/reports/{file}"
        shutil.copy(src, dest)
        print(f"✅ Saved report: {file}")

print(f"\n✅ All files saved to Google Drive!")